# Genre Standardization

Netflix and Disney+ use different genre-labeling systems. Their labels also mix genres with formats, audiences, geographic descriptors, themes, and other characteristics.

This notebook creates a documented cross-platform genre taxonomy. Genre-bearing labels will be mapped into broader comparable categories, while non-genre labels will be preserved in the crosswalk but excluded from the primary genre analysis.

Some source labels map to multiple broad genres. For example, `Romantic Comedy` maps to both `Comedy` and `Romance`. After mapping, duplicate title–broad-genre relationships will be removed before calculating genre measures.

The standardized genres are:

1. Action and Adventure
2. Animation and Anime
3. Children and Family
4. Comedy
5. Crime Mystery and Thriller
6. Documentary
7. Drama
8. Faith and Spirituality
9. Horror
10. Music and Performing Arts
11. Reality and Lifestyle
12. Romance
13. Science Fiction and Fantasy
14. Sports

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROCESSED_DATA_DIR = Path("../data/processed")
REFERENCE_DATA_DIR = Path("../data/reference")
OUTPUT_TABLE_DIR = Path("../outputs/tables")

REFERENCE_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

titles = pd.read_csv(
    PROCESSED_DATA_DIR / "streaming_titles_clean.csv",
    parse_dates=["date_added"]
)

raw_genres = pd.read_csv(
    PROCESSED_DATA_DIR / "streaming_titles_by_genre.csv",
    parse_dates=["date_added"]
)

In [3]:
print("Title-level data:", titles.shape)
print("Raw genre data:", raw_genres.shape)
print("Unique titles:", raw_genres["content_id"].nunique())

Title-level data: (10257, 16)
Raw genre data: (23233, 11)
Unique titles: 10257


In [4]:
INCLUDED_MAPPINGS = [
    # Netflix
    (
        "Netflix",
        "Action & Adventure",
        "Genre",
        ["Action and Adventure"],
        "Direct genre equivalent."
    ),
    (
        "Netflix",
        "Anime Features",
        "Subgenre",
        ["Animation and Anime"],
        "Anime movie label."
    ),
    (
        "Netflix",
        "Anime Series",
        "Subgenre",
        ["Animation and Anime"],
        "Anime television label."
    ),
    (
        "Netflix",
        "Children & Family Movies",
        "Audience and genre",
        ["Children and Family"],
        "Family-oriented movie label."
    ),
    (
        "Netflix",
        "Comedies",
        "Genre",
        ["Comedy"],
        "Plural form of Comedy."
    ),
    (
        "Netflix",
        "Crime TV Shows",
        "Subgenre",
        ["Crime Mystery and Thriller"],
        "Crime television label."
    ),
    (
        "Netflix",
        "Documentaries",
        "Genre",
        ["Documentary"],
        "Plural form of Documentary."
    ),
    (
        "Netflix",
        "Docuseries",
        "Format and genre",
        ["Documentary"],
        "Episodic documentary content."
    ),
    (
        "Netflix",
        "Dramas",
        "Genre",
        ["Drama"],
        "Plural form of Drama."
    ),
    (
        "Netflix",
        "Faith & Spirituality",
        "Genre and theme",
        ["Faith and Spirituality"],
        "Faith-oriented content."
    ),
    (
        "Netflix",
        "Horror Movies",
        "Genre",
        ["Horror"],
        "Horror movie label."
    ),
    (
        "Netflix",
        "Kids' TV",
        "Audience and genre",
        ["Children and Family"],
        "Television for children."
    ),
    (
        "Netflix",
        "Music & Musicals",
        "Genre",
        ["Music and Performing Arts"],
        "Music and musical content."
    ),
    (
        "Netflix",
        "Reality TV",
        "Format and genre",
        ["Reality and Lifestyle"],
        "Reality programming."
    ),
    (
        "Netflix",
        "Romantic Movies",
        "Genre",
        ["Romance"],
        "Romance movie label."
    ),
    (
        "Netflix",
        "Romantic TV Shows",
        "Genre",
        ["Romance"],
        "Romance television label."
    ),
    (
        "Netflix",
        "Sci-Fi & Fantasy",
        "Genre",
        ["Science Fiction and Fantasy"],
        "Combined science-fiction and fantasy label."
    ),
    (
        "Netflix",
        "Science & Nature TV",
        "Genre and subject",
        ["Documentary"],
        "Usually factual science or nature programming."
    ),
    (
        "Netflix",
        "Sports Movies",
        "Genre and subject",
        ["Sports"],
        "Sports-oriented movies."
    ),
    (
        "Netflix",
        "Stand-Up Comedy",
        "Performance genre",
        ["Comedy"],
        "Stand-up performance content."
    ),
    (
        "Netflix",
        "Stand-Up Comedy & Talk Shows",
        "Mixed genre",
        ["Comedy", "Reality and Lifestyle"],
        "Contains both comedy and talk-show characteristics."
    ),
    (
        "Netflix",
        "Teen TV Shows",
        "Audience",
        ["Children and Family"],
        "Included as youth-oriented programming."
    ),
    (
        "Netflix",
        "Thrillers",
        "Genre",
        ["Crime Mystery and Thriller"],
        "Thriller movie label."
    ),
    (
        "Netflix",
        "TV Action & Adventure",
        "Genre",
        ["Action and Adventure"],
        "Television action and adventure."
    ),
    (
        "Netflix",
        "TV Comedies",
        "Genre",
        ["Comedy"],
        "Television comedy."
    ),
    (
        "Netflix",
        "TV Dramas",
        "Genre",
        ["Drama"],
        "Television drama."
    ),
    (
        "Netflix",
        "TV Horror",
        "Genre",
        ["Horror"],
        "Television horror."
    ),
    (
        "Netflix",
        "TV Mysteries",
        "Genre",
        ["Crime Mystery and Thriller"],
        "Television mystery."
    ),
    (
        "Netflix",
        "TV Sci-Fi & Fantasy",
        "Genre",
        ["Science Fiction and Fantasy"],
        "Television science fiction and fantasy."
    ),
    (
        "Netflix",
        "TV Thrillers",
        "Genre",
        ["Crime Mystery and Thriller"],
        "Television thriller."
    ),

    # Disney+
    (
        "Disney+",
        "Action-Adventure",
        "Genre",
        ["Action and Adventure"],
        "Direct equivalent of action and adventure."
    ),
    (
        "Disney+",
        "Animals & Nature",
        "Genre and subject",
        ["Documentary"],
        "Usually factual animal or nature programming."
    ),
    (
        "Disney+",
        "Animation",
        "Medium and genre",
        ["Animation and Anime"],
        "Animated content."
    ),
    (
        "Disney+",
        "Anime",
        "Subgenre",
        ["Animation and Anime"],
        "Anime content."
    ),
    (
        "Disney+",
        "Biographical",
        "Genre and subject",
        ["Drama"],
        "Mapped to Drama rather than assuming documentary form."
    ),
    (
        "Disney+",
        "Comedy",
        "Genre",
        ["Comedy"],
        "Direct genre equivalent."
    ),
    (
        "Disney+",
        "Coming of Age",
        "Genre and audience",
        ["Children and Family", "Drama"],
        "Youth-oriented stories that commonly use dramatic structure."
    ),
    (
        "Disney+",
        "Concert Film",
        "Performance genre",
        ["Music and Performing Arts"],
        "Filmed musical performance."
    ),
    (
        "Disney+",
        "Crime",
        "Genre",
        ["Crime Mystery and Thriller"],
        "Crime content."
    ),
    (
        "Disney+",
        "Dance",
        "Performance genre",
        ["Music and Performing Arts"],
        "Dance and performance content."
    ),
    (
        "Disney+",
        "Disaster",
        "Subgenre",
        ["Action and Adventure"],
        "Disaster-oriented action or adventure."
    ),
    (
        "Disney+",
        "Documentary",
        "Genre",
        ["Documentary"],
        "Direct genre equivalent."
    ),
    (
        "Disney+",
        "Docuseries",
        "Format and genre",
        ["Documentary"],
        "Episodic documentary content."
    ),
    (
        "Disney+",
        "Drama",
        "Genre",
        ["Drama"],
        "Direct genre equivalent."
    ),
    (
        "Disney+",
        "Family",
        "Audience and genre",
        ["Children and Family"],
        "Family-oriented content."
    ),
    (
        "Disney+",
        "Fantasy",
        "Genre",
        ["Science Fiction and Fantasy"],
        "Fantasy content."
    ),
    (
        "Disney+",
        "Game Show / Competition",
        "Format and genre",
        ["Reality and Lifestyle"],
        "Unscripted competition content."
    ),
    (
        "Disney+",
        "Historical",
        "Genre and subject",
        ["Drama"],
        "Mapped to Drama without assuming documentary form."
    ),
    (
        "Disney+",
        "Kids",
        "Audience",
        ["Children and Family"],
        "Programming intended for children."
    ),
    (
        "Disney+",
        "Lifestyle",
        "Genre and subject",
        ["Reality and Lifestyle"],
        "Lifestyle-oriented programming."
    ),
    (
        "Disney+",
        "Medical",
        "Subgenre",
        ["Drama"],
        "Medical drama or narrative content."
    ),
    (
        "Disney+",
        "Music",
        "Genre",
        ["Music and Performing Arts"],
        "Music-oriented content."
    ),
    (
        "Disney+",
        "Musical",
        "Genre",
        ["Music and Performing Arts"],
        "Musical content."
    ),
    (
        "Disney+",
        "Mystery",
        "Genre",
        ["Crime Mystery and Thriller"],
        "Mystery content."
    ),
    (
        "Disney+",
        "Parody",
        "Subgenre",
        ["Comedy"],
        "Parody is treated as comedy."
    ),
    (
        "Disney+",
        "Police/Cop",
        "Subgenre",
        ["Crime Mystery and Thriller"],
        "Police and crime-oriented content."
    ),
    (
        "Disney+",
        "Reality",
        "Format and genre",
        ["Reality and Lifestyle"],
        "Reality programming."
    ),
    (
        "Disney+",
        "Romance",
        "Genre",
        ["Romance"],
        "Direct genre equivalent."
    ),
    (
        "Disney+",
        "Romantic Comedy",
        "Mixed genre",
        ["Comedy", "Romance"],
        "Combines comedy and romance."
    ),
    (
        "Disney+",
        "Science Fiction",
        "Genre",
        ["Science Fiction and Fantasy"],
        "Science-fiction content."
    ),
    (
        "Disney+",
        "Soap Opera / Melodrama",
        "Mixed genre",
        ["Drama", "Romance"],
        "Combines dramatic and romantic conventions."
    ),
    (
        "Disney+",
        "Sports",
        "Genre and subject",
        ["Sports"],
        "Sports-oriented content."
    ),
    (
        "Disney+",
        "Spy/Espionage",
        "Mixed subgenre",
        [
            "Action and Adventure",
            "Crime Mystery and Thriller"
        ],
        "Commonly overlaps action and thriller genres."
    ),
    (
        "Disney+",
        "Superhero",
        "Mixed subgenre",
        [
            "Action and Adventure",
            "Science Fiction and Fantasy"
        ],
        "Commonly combines action with speculative elements."
    ),
    (
        "Disney+",
        "Survival",
        "Subgenre",
        ["Action and Adventure"],
        "Survival-oriented adventure."
    ),
    (
        "Disney+",
        "Talk Show",
        "Format and genre",
        ["Reality and Lifestyle"],
        "Unscripted talk programming."
    ),
    (
        "Disney+",
        "Thriller",
        "Genre",
        ["Crime Mystery and Thriller"],
        "Thriller content."
    ),
    (
        "Disney+",
        "Travel",
        "Genre and subject",
        ["Reality and Lifestyle"],
        "Travel and lifestyle programming."
    ),
    (
        "Disney+",
        "Variety",
        "Format and genre",
        ["Reality and Lifestyle"],
        "Unscripted variety programming."
    ),
    (
        "Disney+",
        "Western",
        "Subgenre",
        ["Action and Adventure"],
        "Mapped to the broader action and adventure category."
    )
]

In [5]:
EXCLUDED_LABELS = [
    # Netflix
    (
        "Netflix",
        "British TV Shows",
        "Geography",
        "Country or regional descriptor rather than a genre."
    ),
    (
        "Netflix",
        "Classic & Cult TV",
        "Characteristic",
        "Describes age or cultural status rather than genre."
    ),
    (
        "Netflix",
        "Classic Movies",
        "Characteristic",
        "Describes age rather than genre."
    ),
    (
        "Netflix",
        "Cult Movies",
        "Characteristic",
        "Describes cultural status rather than genre."
    ),
    (
        "Netflix",
        "Independent Movies",
        "Production characteristic",
        "Describes production context rather than genre."
    ),
    (
        "Netflix",
        "International Movies",
        "Geography",
        "International origin is not a storytelling genre."
    ),
    (
        "Netflix",
        "International TV Shows",
        "Geography",
        "International origin is not a storytelling genre."
    ),
    (
        "Netflix",
        "Korean TV Shows",
        "Geography",
        "Country descriptor rather than genre."
    ),
    (
        "Netflix",
        "LGBTQ Movies",
        "Theme and representation",
        "Identity representation should not be treated as a genre."
    ),
    (
        "Netflix",
        "Movies",
        "Format",
        "Generic content format."
    ),
    (
        "Netflix",
        "Spanish-Language TV Shows",
        "Language",
        "Language descriptor rather than genre."
    ),
    (
        "Netflix",
        "TV Shows",
        "Format",
        "Generic content format."
    ),

    # Disney+
    (
        "Disney+",
        "Anthology",
        "Format",
        "Narrative structure rather than genre."
    ),
    (
        "Disney+",
        "Buddy",
        "Theme",
        "Relationship structure rather than a reliable genre."
    ),
    (
        "Disney+",
        "Movies",
        "Format",
        "Generic content format."
    ),
    (
        "Disney+",
        "Series",
        "Format",
        "Generic content format."
    )
]

In [7]:
crosswalk_rows = []

for (
    platform,
    raw_label,
    label_type,
    broad_genres,
    mapping_note
) in INCLUDED_MAPPINGS:
    for broad_genre in broad_genres:
        crosswalk_rows.append({
            "platform": platform,
            "raw_label": raw_label,
            "label_type": label_type,
            "broad_genre": broad_genre,
            "include_in_analysis": True,
            "mapping_note": mapping_note
        })

for (
    platform,
    raw_label,
    label_type,
    mapping_note
) in EXCLUDED_LABELS:
    crosswalk_rows.append({
        "platform": platform,
        "raw_label": raw_label,
        "label_type": label_type,
        "broad_genre": pd.NA,
        "include_in_analysis": False,
        "mapping_note": mapping_note
    })

genre_crosswalk = pd.DataFrame(crosswalk_rows)

display(
    genre_crosswalk.sort_values(
        ["platform", "raw_label", "broad_genre"]
    )
)

,platform,raw_label,label_type,broad_genre,include_in_analysis,mapping_note
31,Disney+,Action-Adventure,Genre,Action and Adventure,True,Direct equivalent of action and adventure.
32,Disney+,Animals & Nature,Genre and subject,Documentary,True,Usually factual animal or nature programming.
33,Disney+,Animation,Medium and genre,Animation and Anime,True,Animated content.
34,Disney+,Anime,Subgenre,Animation and Anime,True,Anime content.
88,Disney+,Anthology,Format,NaN,False,Narrative structure rather than genre.
...,...,...,...,...,...,...
29,Netflix,TV Sci-Fi & Fantasy,Genre,Science Fiction and Fantasy,True,Television science fiction and fantasy.
87,Netflix,TV Shows,Format,NaN,False,Generic content format.
30,Netflix,TV Thrillers,Genre,Crime Mystery and Thriller,True,Television thriller.
22,Netflix,Teen TV Shows,Audience,Children and Family,True,Included as youth-oriented programming.


In [9]:
available_labels = (
    raw_genres[
        ["platform", "genre"]
    ]
    .drop_duplicates()
    .rename(columns={"genre": "raw_label"})
)
documented_labels = (
    genre_crosswalk[
        ["platform", "raw_label"]
    ]
    .drop_duplicates()
)

missing_from_crosswalk = (
    available_labels
    .merge(
        documented_labels,
        on=["platform", "raw_label"],
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

display(missing_from_crosswalk)

extra_crosswalk_labels = (
    documented_labels
    .merge(
        available_labels,
        on=["platform", "raw_label"],
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

display(extra_crosswalk_labels)



,platform,raw_label


,platform,raw_label


In [11]:
duplicate_mappings = genre_crosswalk.duplicated(
    subset=[
        "platform",
        "raw_label",
        "broad_genre"
    ]
).sum()

print("Duplicate crosswalk mappings:", duplicate_mappings)

included_without_genre = genre_crosswalk[
    genre_crosswalk["include_in_analysis"]
    & genre_crosswalk["broad_genre"].isna()
]

excluded_with_genre = genre_crosswalk[
    ~genre_crosswalk["include_in_analysis"]
    & genre_crosswalk["broad_genre"].notna()
]

print(
    "Included rows without a broad genre:",
    len(included_without_genre)
)

print(
    "Excluded rows with a broad genre:",
    len(excluded_with_genre)
)

Duplicate crosswalk mappings: 0
Included rows without a broad genre: 0
Excluded rows with a broad genre: 0


In [12]:
QUESTIONABLE_LABELS = [
    "Biographical",
    "Historical",
    "Coming of Age",
    "Teen TV Shows",
    "Superhero",
    "Travel",
    "Variety",
    "Buddy",
    "Stand-Up Comedy & Talk Shows"
]

questionable_samples = (
    raw_genres[
        raw_genres["genre"].isin(
            QUESTIONABLE_LABELS
        )
    ]
    .sort_values(
        ["platform", "genre", "title"]
    )
    .groupby(
        ["platform", "genre"],
        group_keys=False
    )
    .head(5)
)

display(
    questionable_samples[
        [
            "platform",
            "genre",
            "title",
            "type",
            "release_year"
        ]
    ]
)


,platform,genre,title,type,release_year
19333,Disney+,Biographical,Becoming Cousteau,Movie,2021
19916,Disney+,Biographical,Being the Queen,Movie,2020
19451,Disney+,Biographical,Bob Ballard: An Explorer's Life,Movie,2021
19943,Disney+,Biographical,Cesar Millan: The Real Story,Movie,2012
20283,Disney+,Biographical,Clouds,Movie,2020
23227,Disney+,Buddy,Bend It Like Beckham,Movie,2003
21259,Disney+,Buddy,Best Friends Whenever,TV Show,2015
21310,Disney+,Buddy,Buffalo Dreams,Movie,2005
20025,Disney+,Buddy,Car SOS,TV Show,2012
19680,Disney+,Buddy,Chip 'n' Dale: Park Life,TV Show,2021


In [13]:
included_crosswalk = genre_crosswalk[
    genre_crosswalk["include_in_analysis"]
].copy()

mapped_genres = raw_genres.merge(
    included_crosswalk[
        [
            "platform",
            "raw_label",
            "broad_genre"
        ]
    ],
    left_on=["platform", "genre"],
    right_on=["platform", "raw_label"],
    how="inner"
)

print("Mapped relationship rows:", len(mapped_genres))
print(
    "Mapped unique titles:",
    mapped_genres["content_id"].nunique()
)

Mapped relationship rows: 17619
Mapped unique titles: 10169


In [14]:
STANDARDIZED_GROUP_COLUMNS = [
    "content_id",
    "platform",
    "type",
    "title",
    "release_year",
    "date_added",
    "rating",
    "broad_genre"
]

standardized_genres = (
    mapped_genres
    .groupby(
        STANDARDIZED_GROUP_COLUMNS,
        dropna=False
    )
    .agg(
        source_labels=(
            "raw_label",
            lambda values: "; ".join(
                sorted(set(values))
            )
        )
    )
    .reset_index()
)

duplicate_standardized_pairs = (
    standardized_genres[
        ["content_id", "broad_genre"]
    ]
    .duplicated()
    .sum()
)

print(
    "Duplicate title–broad-genre pairs:",
    duplicate_standardized_pairs
)

Duplicate title–broad-genre pairs: 0


In [15]:
standardized_genres["broad_genre_count"] = (
    standardized_genres
    .groupby("content_id")["broad_genre"]
    .transform("count")
)

standardized_genres["title_credit"] = 1

standardized_genres["fractional_credit"] = (
    1 / standardized_genres["broad_genre_count"]
)

platform_title_totals = (
    titles
    .groupby("platform")
    .agg(total_titles=("content_id", "nunique"))
    .reset_index()
)

mapped_title_totals = (
    standardized_genres
    .groupby("platform")
    .agg(mapped_titles=("content_id", "nunique"))
    .reset_index()
)

mapping_coverage = platform_title_totals.merge(
    mapped_title_totals,
    on="platform",
    how="left"
)

mapping_coverage["mapped_titles"] = (
    mapping_coverage["mapped_titles"]
    .fillna(0)
    .astype(int)
)

mapping_coverage["unmapped_titles"] = (
    mapping_coverage["total_titles"]
    - mapping_coverage["mapped_titles"]
)

mapping_coverage["coverage_percentage"] = (
    mapping_coverage["mapped_titles"]
    / mapping_coverage["total_titles"]
    * 100
).round(2)

display(mapping_coverage)

,platform,total_titles,mapped_titles,unmapped_titles,coverage_percentage
0,Disney+,1450,1444,6,99.59
1,Netflix,8807,8725,82,99.07


In [16]:
mapped_content_ids = set(
    standardized_genres["content_id"]
)

unmapped_titles = titles[
    ~titles["content_id"].isin(
        mapped_content_ids
    )
].copy()

print(
    "Unmapped titles:",
    len(unmapped_titles)
)

display(
    unmapped_titles[
        [
            "content_id",
            "platform",
            "type",
            "title",
            "listed_in"
        ]
    ].head(20)
)

Unmapped titles: 88


,content_id,platform,type,title,listed_in
148,netflix_s149,Netflix,TV Show,HQ Barbers,TV Shows
297,netflix_s298,Netflix,TV Show,Navarasa,TV Shows
308,netflix_s309,Netflix,Movie,American Masters: Inventing David Geffen,Movies
470,netflix_s471,Netflix,Movie,Bridgerton - The Afterparty,Movies
726,netflix_s727,Netflix,TV Show,Metallica: Some Kind of Monster,TV Shows
729,netflix_s730,Netflix,Movie,Bling Empire - The Afterparty,Movies
730,netflix_s731,Netflix,Movie,Cobra Kai - The Afterparty,Movies
732,netflix_s733,Netflix,Movie,To All the Boys: Always and Forever - The Afte...,Movies
759,netflix_s760,Netflix,Movie,Fireplace 4K: Classic Crackling Fireplace from...,Movies
760,netflix_s761,Netflix,Movie,Fireplace 4K: Crackling Birchwood from Firepla...,Movies


In [17]:
excluded_crosswalk = genre_crosswalk[
    ~genre_crosswalk["include_in_analysis"]
].copy()

excluded_label_frequency = (
    raw_genres
    .merge(
        excluded_crosswalk[
            ["platform", "raw_label", "label_type"]
        ],
        left_on=["platform", "genre"],
        right_on=["platform", "raw_label"],
        how="inner"
    )
    .groupby(
        ["platform", "raw_label", "label_type"]
    )
    .agg(
        unique_titles=("content_id", "nunique")
    )
    .reset_index()
    .sort_values(
        ["platform", "unique_titles"],
        ascending=[True, False]
    )
)

display(excluded_label_frequency)

,platform,raw_label,label_type,unique_titles
1,Disney+,Buddy,Theme,40
0,Disney+,Anthology,Format,28
2,Disney+,Movies,Format,3
3,Disney+,Series,Format,3
9,Netflix,International Movies,Geography,2752
10,Netflix,International TV Shows,Geography,1351
8,Netflix,Independent Movies,Production characteristic,756
4,Netflix,British TV Shows,Geography,253
14,Netflix,Spanish-Language TV Shows,Language,174
11,Netflix,Korean TV Shows,Geography,151


In [18]:
genre_presence_summary = (
    standardized_genres
    .groupby(["platform", "broad_genre"])
    .agg(
        unique_titles=("content_id", "nunique")
    )
    .reset_index()
)

genre_presence_summary = (
    genre_presence_summary
    .merge(
        platform_title_totals,
        on="platform",
        how="left"
    )
)

genre_presence_summary[
    "title_presence_percentage"
] = (
    genre_presence_summary["unique_titles"]
    / genre_presence_summary["total_titles"]
    * 100
).round(2)


display(
    genre_presence_summary.sort_values(
        ["platform", "title_presence_percentage"],
        ascending=[True, False]
    )
)

,platform,broad_genre,unique_titles,total_titles,title_presence_percentage
2,Disney+,Children and Family,911,1450,62.83
1,Disney+,Animation and Anime,542,1450,37.38
3,Disney+,Comedy,533,1450,36.76
0,Disney+,Action and Adventure,465,1450,32.07
6,Disney+,Drama,359,1450,24.76
5,Disney+,Documentary,346,1450,23.86
10,Disney+,Science Fiction and Fantasy,281,1450,19.38
7,Disney+,Music and Performing Arts,94,1450,6.48
8,Disney+,Reality and Lifestyle,54,1450,3.72
11,Disney+,Sports,43,1450,2.97


In [19]:
genre_fractional_summary = (
    standardized_genres
    .groupby(["platform", "broad_genre"])
    .agg(
        fractional_titles=("fractional_credit", "sum")
    )
    .reset_index()
)

genre_fractional_summary[
    "mapped_fractional_total"
] = (
    genre_fractional_summary
    .groupby("platform")["fractional_titles"]
    .transform("sum")
)

genre_fractional_summary[
    "fractional_share_percentage"
] = (
    genre_fractional_summary["fractional_titles"]
    / genre_fractional_summary[
        "mapped_fractional_total"
    ]
    * 100
).round(2)

display(
    genre_fractional_summary
    .groupby("platform")[
        "fractional_share_percentage"
    ]
    .sum()
    .round(2)
)

platform
Disney+    100.00
Netflix     99.98
Name: fractional_share_percentage, dtype: float64

In [20]:
GENRE_CROSSWALK_PATH = (
    REFERENCE_DATA_DIR / "genre_crosswalk.csv"
)

genre_crosswalk.to_csv(
    GENRE_CROSSWALK_PATH,
    index=False
)

STANDARDIZED_GENRE_PATH = (
    PROCESSED_DATA_DIR
    / "streaming_titles_by_standardized_genre.csv"
)

standardized_genres.to_csv(
    STANDARDIZED_GENRE_PATH,
    index=False
)

mapping_coverage.to_csv(
    OUTPUT_TABLE_DIR
    / "genre_mapping_coverage.csv",
    index=False
)

genre_presence_summary.to_csv(
    OUTPUT_TABLE_DIR
    / "genre_presence_summary.csv",
    index=False
)

genre_fractional_summary.to_csv(
    OUTPUT_TABLE_DIR
    / "genre_fractional_summary.csv",
    index=False
)

excluded_label_frequency.to_csv(
    OUTPUT_TABLE_DIR
    / "excluded_genre_label_frequency.csv",
    index=False
)

OUTPUT_FILES = [
    GENRE_CROSSWALK_PATH,
    STANDARDIZED_GENRE_PATH,
    OUTPUT_TABLE_DIR / "genre_mapping_coverage.csv",
    OUTPUT_TABLE_DIR / "genre_presence_summary.csv",
    OUTPUT_TABLE_DIR / "genre_fractional_summary.csv",
    OUTPUT_TABLE_DIR / "excluded_genre_label_frequency.csv"
]

for file_path in OUTPUT_FILES:
    print(file_path.name, "exists:", file_path.exists())

genre_crosswalk.csv exists: True
streaming_titles_by_standardized_genre.csv exists: True
genre_mapping_coverage.csv exists: True
genre_presence_summary.csv exists: True
genre_fractional_summary.csv exists: True
excluded_genre_label_frequency.csv exists: True


## Genre Standardization Results

- Every original platform-genre label was documented in the crosswalk.
- Genre-bearing labels were mapped into 14 standardized broad genres.
- Format, geographic, language, theme, and production-characteristic labels were documented but excluded from the primary genre comparison.
- Selected mixed labels were mapped to more than one broad genre.
- Duplicate title–broad-genre relationships were consolidated.
- Original labels were preserved in `source_labels` for traceability.
- Title-presence percentages were calculated using each platform's complete catalog as the denominator.
- Fractional genre shares were calculated across titles with at least one included broad genre.
- Mapping coverage and excluded-label frequencies were exported for review.
- The standardized title–genre table was exported for visualization in Notebook 7.

These genre results measure catalog composition. They do not measure popularity, audience demand, content quality, or financial performance.